<h2 style="text-align:center;">Optimal Execution<br></h2>
<div style="text-align:center;">Ariel Kalingking</div>
<div style="text-align:center;">akalingking@gmail.com</div>
<p style="text-align:center;">Appendix Python Code</p>

In [121]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
#import matplotlib.ticker as mticker
import logging
#import io
#import base64

imagepath = "../paper/figures/"
matplotlib.use('TkAgg')
logging.getLogger().setLevel(logging.ERROR)
figsize = (6,5)
fontbig = 8
fontmed = 7
fontsmall = 6
# Unset when being built from makefile
plt_inline_enable = False
if plt_inline_enable:
    %matplotlib inline

In [117]:
### Model parameters ###
eta = 0.1                  # Risk aversion for trading (eta > 0)
gamma = 0.01               # Risk aversion for inventory (gamma > 0)
sigma = 0.05               # Volatility of the price process (sigma > 0)
T = 1.0                    # Time horizon (e.g., 1 day)

# Time Discretization
N_t = 200                  # Number of time steps (backward)
t_min, t_max = 0, T        # Boundary condition for time
dt = T / N_t               # Time step size
print("dt:", dt)

# Inventory Discretization
N_q = 100                  # Number of inventory grid points
q_min, q_max = -500, 500    # Boundary condition for inventory size
dq = (q_max - q_min) / N_q  # Inventory step size

dt: 0.005


In [118]:
# Numerical Approximation using derivatives for finding optimal trading rate, v*(q,t).
inventory_grid = np.linspace(q_min, q_max, N_q)
time_grid = np.linspace(t_min, t_max, N_t)
V_grid = np.zeros((N_t, N_q))
V_star = np.zeros((N_t, N_q))

# Set the boundary conditions
V_grid[-1, :] = 0

# Backward Iteration to solve the HJB 
for t in range(N_t-1, 0, -1):
    V_current = V_grid[t].copy() 

    # --- Numerical Approximation of Derivatives ---
    # We need dV/dq at time t (V_t) to solve for V at t-1.
    # Using central difference for dV/dq requires values from adjacent q points.
    # For boundary q points, we'll use forward/backward differences.
    # dV/dq for V_current (dV/dq at time t_i)
    dV_dq_t = np.zeros(N_q)
    dV_dq_t[1:-1] = (V_current[2:] - V_current[:-2]) / (2 * dq)
    dV_dq_t[0] = (V_current[1] - V_current[0]) / dq
    dV_dq_t[-1] = (V_current[-1] - V_current[-2]) / dq
    
    # Recover optimal trading rate v*(q,t) for the current t.
    # v^*(q, t) = (1 / (2*eta)) * (dV/dq)
    V_star[t, :] = (1 / (2 * eta)) * dV_dq_t
    
    # Construct the next State Value/Cost Value backward in time V(q, t) -> V(q, t-dt)
    # using the derived PDE dV/dt = [1/(4*eta)](dV/dq)**2 - (gamma*sigma**2q**2)
    """
    # Using the derived form of the HJB equation
    for j in range(N_q):
        q = inventory_grid[j]
        dV_dq = dV_dq_t[j]
        # Calculate the dV/dt = -H_min(q,t)
        dV_dt = (1 / (4 * eta)) * (dV_dq**2) - (gamma * sigma**2 * q**2)
        Update the value function V(q, t):
        # V(q,t) - V(q, t-dt) = - H_min(q, t) * dt
        # V(q, t-dt) = V(q, t) - H_min(q, t) * dt
        V_grid[t-1, j] = V_current[j] - dV_dt * dt
    """
    # Construct the next State Value/Cost Value backward in time V(q, t) -> V(q, t-dt)
    # using the Hamiltonian and HJB equation equations
    for j in range(N_q):
        q = inventory_grid[j]
        dV_dq = dV_dq_t[j]
        # Calculate the Hamiltonian
        # -dVdt = min_{v}{f(v,t) + dVdq*v}
        # where: f(v,t)=eta*v^2 + gamma*sigma^2*q^2
        dV_dt = -(eta * (-V_star[t, j])**2 + gamma*sigma**2*q**2 - V_star[t, j]*dV_dq)
        ## Update the value function V(q, t):
        # V(q,t) - V(q, t-dt) = - H_min(q, t) * dt
        # V(q, t-dt) = V(q, t) - H_min(q, t) * dt
        V_grid[t-1, j] = V_current[j] - dV_dt * dt

In [119]:
# Plot approximation of V(q,t)
X, Y = np.meshgrid(time_grid, inventory_grid)
Z = V_grid.T
assert np.shape(Z)[0] == np.shape(X)[0]
assert np.shape(Z)[1] == np.shape(Y)[1]
fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')
ax.set_xlabel('Time, $\Delta t$', fontsize=fontmed)
ax.set_ylabel('Inventory, $q$', fontsize=fontmed)
ax.set_zlabel('Cost-to-go, $V(q,t)$', fontsize=fontmed)
ax.set_title('State Value, $V(q,t)$\n$Numerical\;Approx$', fontsize=fontbig, fontweight="bold", y=.99)
ax.tick_params(axis='both', labelcolor="black", labelsize=fontsmall)
ax.invert_xaxis()
ax.set_box_aspect(aspect=None, zoom=0.9)
plt.tight_layout()
plt.savefig(imagepath+"/value_function_numerical.pdf")
if plt_inline_enable:
    plt.show()

In [120]:
### Plot approximation of v*(q,t) ####
X, Y = np.meshgrid(time_grid[1:], inventory_grid)
Z = V_star[1:].T
assert np.shape(Z)[0] == np.shape(X)[0]
assert np.shape(Z)[1] == np.shape(Y)[1]

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')
ax.set_xlabel('Time, $\Delta t$', fontsize=fontbig)
ax.set_ylabel('Inventory, $q$', fontsize=fontbig)
ax.set_zlabel('$v^*(q,t)$', fontsize=fontbig)
ax.set_title('Trading Rate, $v^*(q,t)$\n${Numerical\;Approx}$', fontsize=fontbig, fontweight="bold", y=.99)
ax.tick_params(axis='both', labelcolor="black", labelsize=fontsmall)
ax.invert_yaxis()
ax.view_init(elev=30, azim=120) # Rotate
ax.set_box_aspect(aspect=None, zoom=0.9)
plt.tight_layout()
plt.savefig(imagepath+"/control_function_numerical.pdf")
if plt_inline_enable:
    plt.show()